# Crop & mask

Restrict a raster to an area of interest:

- **`crop(mask=vector)`** — clip to polygon geometry (everything outside becomes no-data).
- **`crop(bbox=..., epsg=...)`** — clip to a rectangular bounding box.
- **`get_mask()`** — the boolean valid-data mask (True where a cell holds real data).

## Setup

In [ ]:
import os

os.environ['MPLBACKEND'] = 'Agg'  # never trigger an interactive backend

import tempfile
from pathlib import Path

import numpy as np


def _find_data():
    for base in [Path.cwd(), *Path.cwd().parents]:
        cand = base / 'tests' / 'data'
        if cand.is_dir():
            return cand.resolve()
    raise FileNotFoundError('Could not locate tests/data from ' + str(Path.cwd()))


DATA = _find_data()
WORK = Path(tempfile.mkdtemp(prefix='pyramids-ops-'))
DATA.is_dir(), WORK.is_dir()

In [ ]:
from pyramids.dataset import Dataset
from pyramids.feature import FeatureCollection

ds = Dataset.read_file(str(DATA / 'acc4000.tif'))
polys = FeatureCollection.read_file(str(DATA / 'coello_polygons.geojson'))
ds.shape, ds.epsg, (len(polys), polys.epsg)

## Crop to polygons — `crop(mask=...)`

The raster shrinks to the polygons' envelope; cells outside the polygons are set to no-data.

In [ ]:
clipped = ds.crop(mask=polys)
clipped.shape, clipped.epsg

## Crop to a bounding box — `crop(bbox=...)`

Keep only the inner half of the raster's extent.

In [ ]:
xmin, ymin, xmax, ymax = ds.bbox
dx, dy = (xmax - xmin) / 4, (ymax - ymin) / 4
inner = ds.crop(bbox=(xmin + dx, ymin + dy, xmax - dx, ymax - dy), epsg=ds.epsg)
inner.shape

## The valid-data mask — `get_mask`

A GDAL-style mask: **nonzero (255)** where the cell holds data, **0** where it is no-data.

In [ ]:
mask = ds.get_mask()
valid = int((mask > 0).sum())
mask.shape, valid, 'valid of', mask.size

## Notes

- `crop` accepts a `FeatureCollection` or a GeoDataFrame as the mask; pass `touch=False` to
  keep only cells whose centre falls inside the geometry.
- See also: [Reproject / resample / align](reproject-resample-align.ipynb),
  [Zonal statistics](zonal-statistics.ipynb).